# Lab 21 — RNN, LSTM, and GRU Models for Time Series with TensorFlow

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-21-rnn-lstm-gru-lab.ipynb)

This lab is designed for independent study. It explains the mathematical ideas before each programming step and then asks you to interpret the results.

## Learning goals

By the end of this lab, you should be able to:

1. Convert a time series into a supervised sequence-learning dataset.
2. Explain why a recurrent neural network processes observations in temporal order.
3. Build SimpleRNN, LSTM, and GRU forecasting models in TensorFlow/Keras.
4. Compare recurrent models with naive forecasting baselines.
5. Diagnose model performance using horizon-wise errors and forecast plots.
6. Understand why LSTM and GRU gates help with longer-range dependence.

We use TensorFlow/Keras because it is available in Google Colab and gives a concise way to build recurrent sequence models.

## 1. Mathematical background: recurrent sequence modeling

A feedforward neural network maps a fixed vector $x$ directly to a prediction $\hat y$. A recurrent neural network processes a sequence one observation at a time.

For a univariate time series window

$$
(x_{t-L+1}, x_{t-L+2}, \ldots, x_t),
$$

a simple recurrent neural network forms hidden states

$$
h_s = \phi(W_x x_s + W_h h_{s-1} + b_h),
$$

where:

- $h_s$ is the hidden state at time $s$,
- $W_x$ maps the current input into hidden coordinates,
- $W_h$ maps the previous hidden state into the next hidden state,
- $\phi$ is usually a nonlinear activation such as $\tanh$.

After processing the input window, the final hidden state can be used to predict future values:

$$
(\hat x_{t+1}, \ldots, \hat x_{t+H}) = g(h_t).
$$

The input length $L$ controls how much history the model sees. The forecast horizon $H$ controls how far ahead we predict.

## 2. Why LSTM and GRU were introduced

A vanilla RNN repeatedly multiplies by hidden-to-hidden weights. During backpropagation through time, gradients may shrink or explode as they pass through many time steps. This makes it difficult to learn long-range dependence.

LSTM and GRU models introduce **gates**. Gates are learned numbers between 0 and 1 that control how information flows.

Conceptually:

- A **forget gate** decides what old information should be kept.
- An **input/update gate** decides what new information should enter the state.
- An **output gate** decides what part of the state should be exposed to the prediction layer.

A GRU is a simpler gated recurrent model. In practice, LSTM and GRU often train more reliably than a vanilla RNN when the useful signal is spread over many time steps.

## 3. Programming setup

This lab uses TensorFlow/Keras. Google Colab normally has TensorFlow installed. If you run this notebook locally and TensorFlow is missing, install it first with:

```bash
pip install tensorflow
```

Then restart the kernel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "TensorFlow is not installed. In Google Colab it is usually available. "
        "For local use, run: pip install tensorflow"
    ) from err

from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(7339)
tf.keras.utils.set_random_seed(7339)

# Keep training reasonably reproducible and Colab-friendly.
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("TensorFlow version:", tf.__version__)

## 4. Simulate a time series with short and long memory

We will simulate a series with several components:

1. a slow trend,
2. a daily-like seasonal cycle,
3. a weekly-like seasonal cycle,
4. autoregressive dependence,
5. occasional shocks whose effects decay slowly.

The slowly decaying shocks create a simple form of long memory. This is useful for comparing SimpleRNN, LSTM, and GRU models.

In [ ]:
def simulate_sequence(n=2600, seed=7339):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    trend = 0.0008 * t
    daily = 0.9 * np.sin(2 * np.pi * t / 24)
    weekly = 0.55 * np.sin(2 * np.pi * t / 168)

    # Sparse shocks. Each shock has a slowly decaying effect.
    shock_times = np.arange(180, n, 220)
    shock_amplitudes = rng.normal(1.5, 0.35, size=len(shock_times))
    slow_effect = np.zeros(n)
    for time, amplitude in zip(shock_times, shock_amplitudes):
        decay_length = min(120, n - time)
        slow_effect[time:time + decay_length] += amplitude * np.exp(-np.arange(decay_length) / 45)

    noise = rng.normal(0, 0.20, size=n)
    x = np.zeros(n)
    for i in range(2, n):
        x[i] = (
            0.55 * x[i - 1]
            - 0.12 * x[i - 2]
            + trend[i]
            + daily[i]
            + weekly[i]
            + slow_effect[i]
            + noise[i]
        )

    return pd.DataFrame({
        "time": t,
        "value": x,
        "trend": trend,
        "daily": daily,
        "weekly": weekly,
        "slow_effect": slow_effect,
    })

series = simulate_sequence()
series.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(series["time"], series["value"], linewidth=1)
plt.title("Synthetic time series with seasonality, autoregression, and slowly decaying shocks")
plt.xlabel("time")
plt.ylabel("value")
plt.show()

plt.figure(figsize=(11, 3))
plt.plot(series["time"], series["slow_effect"], linewidth=1)
plt.title("Hidden slowly decaying shock component")
plt.xlabel("time")
plt.ylabel("effect")
plt.show()

### Interpretation checkpoint

Before training any neural network, answer these questions:

1. What patterns are visible by eye?
2. Which patterns are short-range?
3. Which patterns require the model to remember farther into the past?
4. Why might a recurrent model be useful here?

## 5. Convert the series into sequence-learning examples

We choose:

- input width $L = 96$, so each model sees the past 96 observations,
- forecast horizon $H = 24$, so each model predicts the next 24 observations.

For each time $t$, the input is

$$
X_t = (x_{t-L+1}, \ldots, x_t),
$$

and the target is

$$
y_t = (x_{t+1}, \ldots, x_{t+H}).
$$

In TensorFlow, recurrent layers expect input shape:

$$
(\text{number of examples}, \text{time steps}, \text{features}).
$$

For a univariate time series, the number of features is 1.

In [ ]:
def make_windows(values, input_width=96, horizon=24):
    values = np.asarray(values, dtype=np.float32)
    X, y = [], []
    last_start = len(values) - input_width - horizon + 1
    for start in range(last_start):
        end = start + input_width
        X.append(values[start:end])
        y.append(values[end:end + horizon])
    X = np.array(X, dtype=np.float32)[..., None]
    y = np.array(y, dtype=np.float32)
    return X, y

INPUT_WIDTH = 96
HORIZON = 24

values = series["value"].values.astype(np.float32)
X_raw, y_raw = make_windows(values, input_width=INPUT_WIDTH, horizon=HORIZON)

print("X shape:", X_raw.shape)
print("y shape:", y_raw.shape)
print("One input window shape:", X_raw[0].shape)
print("One target shape:", y_raw[0].shape)

## 6. Chronological train/validation/test split and scaling

For time series, we should not randomly shuffle observations before splitting. Random splitting can leak future information into the training set.

We split examples chronologically:

- first 60% for training,
- next 20% for validation,
- final 20% for testing.

We estimate scaling parameters using the training period only.

In [ ]:
n_examples = len(X_raw)
train_end = int(0.60 * n_examples)
val_end = int(0.80 * n_examples)

X_train_raw, y_train_raw = X_raw[:train_end], y_raw[:train_end]
X_val_raw, y_val_raw = X_raw[train_end:val_end], y_raw[train_end:val_end]
X_test_raw, y_test_raw = X_raw[val_end:], y_raw[val_end:]

train_mean = X_train_raw.mean()
train_std = X_train_raw.std()

X_train = (X_train_raw - train_mean) / train_std
X_val = (X_val_raw - train_mean) / train_std
X_test = (X_test_raw - train_mean) / train_std

y_train = (y_train_raw - train_mean) / train_std
y_val = (y_val_raw - train_mean) / train_std
y_test = (y_test_raw - train_mean) / train_std

print("train examples:", len(X_train))
print("validation examples:", len(X_val))
print("test examples:", len(X_test))
print("training mean:", round(float(train_mean), 4))
print("training std:", round(float(train_std), 4))

In [ ]:
def plot_window(index=100):
    input_values = X_train_raw[index, :, 0]
    target_values = y_train_raw[index]
    input_time = np.arange(INPUT_WIDTH)
    target_time = np.arange(INPUT_WIDTH, INPUT_WIDTH + HORIZON)

    plt.figure(figsize=(10, 4))
    plt.plot(input_time, input_values, label="input history")
    plt.plot(target_time, target_values, marker="o", label="future target")
    plt.axvline(INPUT_WIDTH - 1, linestyle="--", linewidth=1)
    plt.title("One supervised learning example")
    plt.xlabel("relative time")
    plt.ylabel("value")
    plt.legend()
    plt.show()

plot_window(index=100)

## 7. Forecasting baselines

A neural model is useful only if it improves over simple baselines.

We compare with two baselines:

1. **Persistence**: predict every future value by the last observed value.
2. **Seasonal naive**: repeat the most recent 24-step pattern.

For hourly-like data, the seasonal naive baseline is often strong because yesterday's pattern can be informative for today.

In [ ]:
def persistence_forecast(X):
    last_value = X[:, -1, 0]
    return np.repeat(last_value[:, None], HORIZON, axis=1)


def seasonal_naive_forecast(X, season=24):
    if X.shape[1] < season:
        return persistence_forecast(X)
    return X[:, -season:, 0][:, :HORIZON]


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def horizon_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred), axis=0)


def horizon_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))

baseline_predictions = {
    "persistence": persistence_forecast(X_test),
    "seasonal_naive": seasonal_naive_forecast(X_test, season=24),
}

baseline_rows = []
for name, pred in baseline_predictions.items():
    baseline_rows.append({
        "model": name,
        "MAE_scaled": mae(y_test, pred),
        "RMSE_scaled": rmse(y_test, pred),
    })

pd.DataFrame(baseline_rows)

## 8. TensorFlow datasets

TensorFlow datasets make batching and prefetching easy.

We keep `shuffle=False` for validation and test sets. For training windows, moderate shuffling is acceptable because each window has already been constructed from the chronological series, and the validation/test periods still remain strictly future periods.

In [ ]:
BATCH_SIZE = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train), seed=7339)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

for xb, yb in train_ds.take(1):
    print("batch X shape:", xb.shape)
    print("batch y shape:", yb.shape)

## 9. Build SimpleRNN, LSTM, and GRU models

Each model has the same high-level structure:

1. sequence input,
2. recurrent layer,
3. last hidden state,
4. dense layer predicting the next 24 time steps.

This makes the comparison more meaningful.

In [ ]:
def build_recurrent_model(kind="lstm", units=32, input_width=INPUT_WIDTH, horizon=HORIZON):
    inputs = keras.Input(shape=(input_width, 1), name="input_window")

    if kind == "simple_rnn":
        seq = layers.SimpleRNN(units, return_sequences=True, name="simple_rnn_sequence")(inputs)
    elif kind == "lstm":
        seq = layers.LSTM(units, return_sequences=True, name="lstm_sequence")(inputs)
    elif kind == "gru":
        seq = layers.GRU(units, return_sequences=True, name="gru_sequence")(inputs)
    else:
        raise ValueError("kind must be 'simple_rnn', 'lstm', or 'gru'")

    seq = layers.LayerNormalization(name="sequence_normalization")(seq)
    last = layers.Lambda(lambda z: z[:, -1, :], name="last_hidden_state")(seq)
    dense = layers.Dense(32, activation="relu", name="dense_features")(last)
    outputs = layers.Dense(horizon, name="forecast")(dense)

    model = keras.Model(inputs=inputs, outputs=outputs, name=f"{kind}_forecast_model")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

simple_rnn_model = build_recurrent_model("simple_rnn")
lstm_model = build_recurrent_model("lstm")
gru_model = build_recurrent_model("gru")

lstm_model.summary()

## 10. Train the recurrent models

We use early stopping to avoid unnecessary training and to reduce overfitting.

The training target is scaled. We will convert forecasts back to the original units when plotting.

In [ ]:
def train_model(model, train_ds, val_ds, max_epochs=20):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )
    ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max_epochs,
        callbacks=callbacks,
        verbose=1,
    )
    return history

histories = {}
models = {
    "SimpleRNN": simple_rnn_model,
    "LSTM": lstm_model,
    "GRU": gru_model,
}

for name, model in models.items():
    print("\n" + "=" * 70)
    print("Training", name)
    histories[name] = train_model(model, train_ds, val_ds, max_epochs=20)

In [ ]:
def plot_training_curves(histories):
    plt.figure(figsize=(10, 4))
    for name, history in histories.items():
        plt.plot(history.history["loss"], label=f"{name} train")
        plt.plot(history.history["val_loss"], linestyle="--", label=f"{name} val")
    plt.title("Training and validation loss")
    plt.xlabel("epoch")
    plt.ylabel("MSE loss")
    plt.legend()
    plt.show()

plot_training_curves(histories)

### Interpretation checkpoint

Compare the training curves.

1. Which model trains fastest?
2. Which model has the lowest validation loss?
3. Do any models show signs of overfitting?
4. Does the gated structure of LSTM or GRU appear useful for this data?

## 11. Evaluate on the test period

The test set represents the final chronological block of data. It was not used for training or early stopping.

In [ ]:
def predict_scaled(model, X):
    return model.predict(X, verbose=0)

nn_predictions = {name: predict_scaled(model, X_test) for name, model in models.items()}

rows = []
for name, pred in baseline_predictions.items():
    rows.append({"model": name, "MAE_scaled": mae(y_test, pred), "RMSE_scaled": rmse(y_test, pred)})
for name, pred in nn_predictions.items():
    rows.append({"model": name, "MAE_scaled": mae(y_test, pred), "RMSE_scaled": rmse(y_test, pred)})

results = pd.DataFrame(rows).sort_values("RMSE_scaled")
results

In [ ]:
plt.figure(figsize=(10, 4))
for name, pred in nn_predictions.items():
    plt.plot(np.arange(1, HORIZON + 1), horizon_rmse(y_test, pred), marker="o", label=name)
for name, pred in baseline_predictions.items():
    plt.plot(np.arange(1, HORIZON + 1), horizon_rmse(y_test, pred), linestyle="--", label=name)
plt.title("Horizon-wise RMSE on the test set")
plt.xlabel("forecast horizon")
plt.ylabel("RMSE in scaled units")
plt.legend()
plt.show()

## 12. Plot forecasts in original units

The models were trained on scaled values. To interpret forecasts, we convert them back to the original scale.

In [ ]:
def inverse_scale(z):
    return z * train_std + train_mean


def plot_forecast_example(example_index=50):
    history = inverse_scale(X_test[example_index, :, 0])
    truth = inverse_scale(y_test[example_index])

    input_time = np.arange(INPUT_WIDTH)
    future_time = np.arange(INPUT_WIDTH, INPUT_WIDTH + HORIZON)

    plt.figure(figsize=(11, 5))
    plt.plot(input_time, history, label="input history")
    plt.plot(future_time, truth, marker="o", linewidth=2, label="true future")

    for name, pred_scaled in nn_predictions.items():
        pred = inverse_scale(pred_scaled[example_index])
        plt.plot(future_time, pred, marker="o", alpha=0.8, label=name)

    seasonal_pred = inverse_scale(baseline_predictions["seasonal_naive"][example_index])
    plt.plot(future_time, seasonal_pred, linestyle="--", label="seasonal naive")

    plt.axvline(INPUT_WIDTH - 1, linestyle="--", linewidth=1)
    plt.title("Forecast comparison for one test example")
    plt.xlabel("relative time")
    plt.ylabel("value")
    plt.legend()
    plt.show()

plot_forecast_example(example_index=50)

### Interpretation checkpoint

Use the forecast plot to answer:

1. Which model tracks the direction of change best?
2. Which model captures seasonality best?
3. Which model underreacts or overreacts to recent shocks?
4. Does the best global RMSE model also look best for this particular example?

## 13. Inspect hidden-state trajectories

Because our recurrent layers use `return_sequences=True`, we can inspect the hidden state at every input time step.

The hidden units are not directly interpretable like AR coefficients. Still, plotting them can help us see whether the model reacts to events in the input window.

In [ ]:
def hidden_sequence_model(model, layer_name):
    return keras.Model(inputs=model.input, outputs=model.get_layer(layer_name).output)

lstm_hidden_model = hidden_sequence_model(lstm_model, "lstm_sequence")
hidden_values = lstm_hidden_model.predict(X_test[50:51], verbose=0)[0]

plt.figure(figsize=(11, 4))
for unit in range(min(6, hidden_values.shape[1])):
    plt.plot(hidden_values[:, unit], label=f"unit {unit}")
plt.title("Selected LSTM hidden-state trajectories for one input window")
plt.xlabel("input time step")
plt.ylabel("hidden activation")
plt.legend()
plt.show()

### Interpretation checkpoint

Look at the hidden-state trajectories.

1. Are some units smooth while others change quickly?
2. Do hidden states respond around peaks, troughs, or shocks?
3. Why should we be cautious about over-interpreting hidden units?

## 14. Add calendar features to the recurrent model

So far, the model sees only past values. Recurrent networks can also use multiple features at each time step.

For example, if the series is hourly, time-of-day and time-of-week features can be represented using sine and cosine terms:

$$
\sin(2\pi t / 24), \quad \cos(2\pi t / 24),
$$

and similarly for weekly seasonality.

These features encode periodic position without creating artificial jumps between the end and beginning of a cycle.

In [ ]:
def build_feature_matrix(df):
    t = df["time"].values.astype(np.float32)
    value = df["value"].values.astype(np.float32)
    features = np.column_stack([
        value,
        np.sin(2 * np.pi * t / 24),
        np.cos(2 * np.pi * t / 24),
        np.sin(2 * np.pi * t / 168),
        np.cos(2 * np.pi * t / 168),
    ]).astype(np.float32)
    return features


def make_feature_windows(features, target_values, input_width=96, horizon=24):
    X, y = [], []
    last_start = len(target_values) - input_width - horizon + 1
    for start in range(last_start):
        end = start + input_width
        X.append(features[start:end, :])
        y.append(target_values[end:end + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

feature_raw = build_feature_matrix(series)
Xf_raw, yf_raw = make_feature_windows(feature_raw, values, INPUT_WIDTH, HORIZON)

Xf_train_raw, yf_train_raw = Xf_raw[:train_end], yf_raw[:train_end]
Xf_val_raw, yf_val_raw = Xf_raw[train_end:val_end], yf_raw[train_end:val_end]
Xf_test_raw, yf_test_raw = Xf_raw[val_end:], yf_raw[val_end:]

# Scale only the value feature using training data. Sine/cosine features already lie in [-1, 1].
value_mean = Xf_train_raw[:, :, 0].mean()
value_std = Xf_train_raw[:, :, 0].std()

Xf_train = Xf_train_raw.copy()
Xf_val = Xf_val_raw.copy()
Xf_test = Xf_test_raw.copy()

for Xf in [Xf_train, Xf_val, Xf_test]:
    Xf[:, :, 0] = (Xf[:, :, 0] - value_mean) / value_std

yf_train = (yf_train_raw - value_mean) / value_std
yf_val = (yf_val_raw - value_mean) / value_std
yf_test = (yf_test_raw - value_mean) / value_std

print("Multifeature X shape:", Xf_train.shape)
print("Target shape:", yf_train.shape)

In [ ]:
train_feature_ds = (
    tf.data.Dataset.from_tensor_slices((Xf_train, yf_train))
    .shuffle(buffer_size=len(Xf_train), seed=7339)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_feature_ds = (
    tf.data.Dataset.from_tensor_slices((Xf_val, yf_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

def build_multifeature_gru(input_width=INPUT_WIDTH, n_features=5, horizon=HORIZON, units=32):
    inputs = keras.Input(shape=(input_width, n_features), name="multifeature_input")
    seq = layers.GRU(units, return_sequences=True, name="gru_multifeature_sequence")(inputs)
    seq = layers.LayerNormalization(name="multifeature_normalization")(seq)
    last = layers.Lambda(lambda z: z[:, -1, :], name="last_hidden_state")(seq)
    dense = layers.Dense(32, activation="relu", name="dense_features")(last)
    outputs = layers.Dense(horizon, name="forecast")(dense)
    model = keras.Model(inputs, outputs, name="multifeature_gru_forecast_model")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model

multifeature_gru = build_multifeature_gru()
history_feature_gru = train_model(multifeature_gru, train_feature_ds, val_feature_ds, max_epochs=20)

In [ ]:
feature_pred = multifeature_gru.predict(Xf_test, verbose=0)
comparison = pd.concat([
    results,
    pd.DataFrame([{
        "model": "GRU_with_calendar_features",
        "MAE_scaled": mae(yf_test, feature_pred),
        "RMSE_scaled": rmse(yf_test, feature_pred),
    }])
], ignore_index=True).sort_values("RMSE_scaled")

comparison

### Interpretation checkpoint

Calendar features may help because the model does not need to infer periodic position only from the recent values.

Answer:

1. Did the feature-based GRU improve performance?
2. If it improved, was the improvement large or small?
3. Why might a recurrent model still benefit from explicit time features?

## 15. Optional long-memory toy experiment

The following optional experiment creates a classification task where the label depends on a signal near the beginning of the sequence. This is a clean way to see why long-range dependence can be hard for a vanilla RNN.

To run it, change `RUN_LONG_MEMORY_DEMO` to `True`.

In [ ]:
RUN_LONG_MEMORY_DEMO = False

if RUN_LONG_MEMORY_DEMO:
    rng = np.random.default_rng(7339)
    n_samples = 3000
    length = 80

    X_cls = rng.normal(0, 0.5, size=(n_samples, length, 1)).astype(np.float32)
    labels = rng.integers(0, 2, size=n_samples)

    # The label is encoded only at the first time step.
    X_cls[:, 0, 0] += np.where(labels == 1, 1.0, -1.0)
    y_cls = labels.astype(np.float32)

    split = int(0.8 * n_samples)
    Xc_train, Xc_test = X_cls[:split], X_cls[split:]
    yc_train, yc_test = y_cls[:split], y_cls[split:]

    def build_classifier(kind="simple_rnn", units=24):
        inputs = keras.Input(shape=(length, 1))
        if kind == "simple_rnn":
            x = layers.SimpleRNN(units)(inputs)
        elif kind == "lstm":
            x = layers.LSTM(units)(inputs)
        elif kind == "gru":
            x = layers.GRU(units)(inputs)
        outputs = layers.Dense(1, activation="sigmoid")(x)
        model = keras.Model(inputs, outputs)
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        return model

    classifiers = {kind: build_classifier(kind) for kind in ["simple_rnn", "lstm", "gru"]}
    for name, clf in classifiers.items():
        print("Training", name)
        clf.fit(Xc_train, yc_train, validation_split=0.2, epochs=6, batch_size=64, verbose=0)
        loss, acc = clf.evaluate(Xc_test, yc_test, verbose=0)
        print(f"{name:>10s} test accuracy: {acc:.3f}")
else:
    print("Optional long-memory demo skipped. Set RUN_LONG_MEMORY_DEMO = True to run it.")

## 16. Exercises

### Exercise 1. Change the input length

Try `INPUT_WIDTH = 48`, `96`, and `168`. Recreate the windows and retrain at least one model.

Questions:

1. Does longer history improve forecasting?
2. Does longer history make training slower?
3. Does the best input width depend on the forecast horizon?

### Exercise 2. Compare model sizes

Try `units = 16`, `32`, and `64`.

Questions:

1. Does a larger model always perform better?
2. Does the validation curve show overfitting?
3. Which model size would you choose for a production forecast?

### Exercise 3. One-step versus multi-step forecasting

Set `HORIZON = 1` and compare the results with `HORIZON = 24`.

Questions:

1. Why is one-step forecasting easier?
2. Which horizons contribute most to the error in the 24-step task?
3. How could you train a model differently for long-horizon forecasting?

### Exercise 4. Residual diagnostics

For the best model, compute residuals on the test set:

$$
e_{i,h} = y_{i,h} - \hat y_{i,h}.
$$

Plot residual histograms for horizon 1, horizon 12, and horizon 24.

Questions:

1. Are residuals centered near zero?
2. Does residual variance grow with horizon?
3. Are there systematic errors around shocks?

## 17. Mini-projects

Choose one mini-project.

### Mini-project A. Recurrent forecasting report

Write a short report comparing SimpleRNN, LSTM, GRU, and seasonal naive forecasting.

Your report should include:

1. description of the data-generating process,
2. model architectures,
3. validation and test metrics,
4. horizon-wise error plots,
5. forecast plots,
6. a conclusion about when recurrent models are worth using.

### Mini-project B. Feature-enhanced recurrent forecasting

Add more features to the multifeature GRU, such as rolling means or lagged seasonal indicators.

Important: compute any rolling feature without using future data.

### Mini-project C. Long-memory stress test

Modify the simulation so shocks have longer or shorter decay. Study when LSTM/GRU models outperform SimpleRNN.

## 18. AI-assisted study prompts

Use these prompts to guide your own study. Do not simply paste the answer into your report; check every claim with code and plots.

1. "Explain the difference between SimpleRNN, LSTM, and GRU in the context of time series forecasting."
2. "Why can random train/test splitting cause leakage in time series forecasting?"
3. "Given horizon-wise RMSE values, how should I interpret errors that increase with horizon?"
4. "Help me design a residual diagnostic plan for a neural time series forecasting model."
5. "What is the difference between direct multi-step forecasting and recursive one-step forecasting?"

## Summary

In this lab, you converted a time series into supervised sequence windows, trained SimpleRNN, LSTM, and GRU models in TensorFlow/Keras, compared them with strong baselines, inspected hidden states, and tested the value of calendar features. The main lesson is that recurrent models are useful when temporal order and hidden state matter, but they should always be evaluated against simple forecasting baselines.